# InsightFlow AI — Multi-Task Ticket Classifier (REAL data, Kaggle training)

Fine-tunes one shared transformer encoder with **three heads** on the **real Bitext customer-support dataset** (~26.9k rows):
- `category`  — real Bitext label (11 classes)
- `intent`    — real Bitext label (27 classes)
- `sentiment` — weak-labeled by a pretrained sentiment model, then **balanced** (see below)

`team` is rule-based routing from category (done in the agent, not learned). `urgency` was dropped — the real data has no honest urgency label.

**Sentiment balancing (v3):** the teacher model labels almost nothing `Positive`, so we add ~700 *domain-neutral* positive reviews (pure praise, no order/delivery/checkout words — those leaked transactional vocabulary into the Positive class in an earlier version and made it over-fire). Positive reviews are labeled `FEEDBACK` / `review`. The sentiment loss uses inverse-frequency class weights **capped at 5x** so the minority class is learned without being over-predicted.

**Calibration:** label smoothing (0.1) keeps the model from being 100%-confident, so its mistakes are more likely to land in the agent's low-confidence triage queue.

### Setup before Run All
1. **Add the dataset:** right panel -> *Add Input* -> search **Bitext Gen AI Chatbot Customer Support** -> Add. (Owner `bitext`.)
2. **Internet: ON** — needed to download the base + sentiment models (HF fallback for the data too).
3. **Accelerator: GPU**.

Then *Run All*. The last cells save and zip `insightflow_ticket_classifier.zip`. Unzip it into `models/insightflow_ticket_classifier/` in the repo.

In [ ]:
import platform, torch, transformers
print('python      :', platform.python_version())
print('torch       :', torch.__version__)
print('transformers:', transformers.__version__)
print('cuda        :', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Config

In [ ]:
import random, numpy as np, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

BASE_MODEL       = 'distilbert-base-uncased'
SENTIMENT_MODEL  = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
MAX_LEN     = 128
BATCH_SIZE  = 32
EPOCHS      = 3
LR          = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_ROWS    = None   # None = use all ~26.9k rows; set e.g. 12000 to train faster
N_POS_AUG   = 700    # domain-neutral positive reviews added to rescue the Positive class
MAX_CLASS_WEIGHT = 5.0   # cap so the minority Positive class is not over-predicted
LABEL_SMOOTHING  = 0.1

SENTIMENTS = ['Negative', 'Neutral', 'Positive']
TASK_WEIGHTS = {'category': 1.0, 'intent': 1.0, 'sentiment': 0.7}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2. Load the real Bitext dataset
Finds the CSV under `/kaggle/input` (any file with `instruction` + `category` columns). Falls back to the HuggingFace Hub if no input dataset is attached.

In [ ]:
import glob
import pandas as pd

def find_bitext_csv():
    for path in glob.glob('/kaggle/input/**/*.csv', recursive=True):
        try:
            head = pd.read_csv(path, nrows=5)
        except Exception:
            continue
        cols = set(c.lower() for c in head.columns)
        if 'instruction' in cols and 'category' in cols:
            return path
    return None

csv_path = find_bitext_csv()
if csv_path is not None:
    print('loading from Kaggle input:', csv_path)
    df = pd.read_csv(csv_path)
else:
    print('no Kaggle input found - downloading Bitext from HuggingFace Hub...')
    from datasets import load_dataset
    ds = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train')
    df = ds.to_pandas()

df.columns = [c.lower() for c in df.columns]
df = df[['instruction', 'category', 'intent']].dropna().reset_index(drop=True)
df = df.rename(columns={'instruction': 'text'})
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() > 0].reset_index(drop=True)
if MAX_ROWS is not None:
    df = df.sample(n=min(MAX_ROWS, len(df)), random_state=SEED).reset_index(drop=True)
print('rows:', len(df))
print('categories:', sorted(df['category'].unique()))
print('num intents:', df['intent'].nunique())
df.head(3)

## 3. Weak-label sentiment

In [ ]:
from transformers import pipeline

sent_pipe = pipeline('sentiment-analysis', model=SENTIMENT_MODEL,
                     device=0 if device == 'cuda' else -1, truncation=True, max_length=256)

MAP = {'negative': 'Negative', 'neutral': 'Neutral', 'positive': 'Positive',
       'label_0': 'Negative', 'label_1': 'Neutral', 'label_2': 'Positive'}

texts = df['text'].tolist()
labels = []
for i in range(0, len(texts), 256):
    out = sent_pipe(texts[i:i + 256], batch_size=64)
    labels += [MAP.get(o['label'].lower(), 'Neutral') for o in out]
    if i % 5120 == 0:
        print('labeled', i + len(out), '/', len(texts))
df['sentiment'] = labels
print('teacher labels:', dict(df['sentiment'].value_counts()))
del sent_pipe
torch.cuda.empty_cache() if device == 'cuda' else None

## 3b. Augment the Positive class (domain-neutral praise)
Pure praise with no order/delivery/checkout/refund vocabulary, so the model does not learn to associate those domains with Positive. Labeled `FEEDBACK` / `review`. Kept distinct from the held-out OOD eval sentences.

In [ ]:
POS_OPENERS = ['Thank you so much', 'Thanks a lot', 'I just wanted to say', 'Huge thanks to your team',
               'I really appreciate it', 'Amazing service', 'I am very happy', 'I am really impressed',
               'What a great experience', 'You folks are awesome', 'Just dropping a note to say',
               'Cannot thank you enough', 'Really pleased', 'Brilliant work from your team']
POS_BODIES = ['your team resolved my issue so quickly', 'the support was outstanding',
              'you made this so easy for me', 'the agent was friendly and helpful',
              'this is exactly what I needed', 'the whole process was effortless',
              'I am delighted with how this turned out', 'the team handled it professionally',
              'support went above and beyond', 'you were incredibly helpful and patient',
              'everything went smoothly', 'the help I got was first class']
POS_CLOSERS = ['keep up the great work', 'you have a customer for life', 'highly recommend you',
               'five stars from me', 'thanks again', 'much appreciated', 'truly grateful',
               'so happy with it', 'cheers', 'really impressed', 'will recommend to friends']

pos_rng = random.Random(SEED + 1)
pos_texts = set()
while len(pos_texts) < N_POS_AUG:
    o = pos_rng.choice(POS_OPENERS); b = pos_rng.choice(POS_BODIES); c = pos_rng.choice(POS_CLOSERS)
    pos_texts.add(o + ', ' + b + '. ' + c + '.')
pos_df = pd.DataFrame({'text': list(pos_texts)})
pos_df['category'] = 'FEEDBACK'
pos_df['intent'] = 'review'
pos_df['sentiment'] = 'Positive'
df = pd.concat([df, pos_df], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print('added', len(pos_df), 'positive reviews')
print('balanced sentiment:', dict(df['sentiment'].value_counts()))

## 4. Label maps and stratified splits

In [ ]:
from sklearn.model_selection import train_test_split

LABELS = {
    'category':  sorted(df['category'].unique().tolist()),
    'intent':    sorted(df['intent'].unique().tolist()),
    'sentiment': SENTIMENTS,
}
TASKS = ['category', 'intent', 'sentiment']
label2id = {t: {lab: i for i, lab in enumerate(LABELS[t])} for t in TASKS}
id2label = {t: {i: lab for i, lab in enumerate(LABELS[t])} for t in TASKS}
print('classes -> category:', len(LABELS['category']), '| intent:', len(LABELS['intent']), '| sentiment:', len(LABELS['sentiment']))

train_df, tmp_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['sentiment'])
val_df, test_df = train_test_split(tmp_df, test_size=0.5, random_state=SEED, stratify=tmp_df['sentiment'])
train_df = train_df.reset_index(drop=True); val_df = val_df.reset_index(drop=True); test_df = test_df.reset_index(drop=True)
print('train/val/test:', len(train_df), len(val_df), len(test_df))

## 5. Tokenization and Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

class TicketDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame['text'].tolist()
        self.labels = {t: [label2id[t][v] for v in frame[t].tolist()] for t in TASKS}
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        item = {'text': self.texts[i]}
        for t in TASKS:
            item[t] = self.labels[t][i]
        return item

def collate(batch):
    enc = tokenizer([b['text'] for b in batch], truncation=True, padding=True,
                    max_length=MAX_LEN, return_tensors='pt')
    for t in TASKS:
        enc[t] = torch.tensor([b[t] for b in batch], dtype=torch.long)
    return enc

train_dl = DataLoader(TicketDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_dl   = DataLoader(TicketDataset(val_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
test_dl  = DataLoader(TicketDataset(test_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
print('batches:', len(train_dl), len(val_dl), len(test_dl))

## 6. Model — shared encoder + multi-task heads

In [ ]:
import json
import torch.nn as nn
from pathlib import Path
from transformers import AutoModel

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

class MultiTaskClassifier(nn.Module):
    def __init__(self, base_model_name, head_labels, dropout=0.1):
        super().__init__()
        self.base_model_name = base_model_name
        self.head_labels = head_labels
        self.encoder = AutoModel.from_pretrained(base_model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.heads = nn.ModuleDict({t: nn.Linear(hidden, len(labs)) for t, labs in head_labels.items()})
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(mean_pool(out.last_hidden_state, attention_mask))
        return {t: head(pooled) for t, head in self.heads.items()}
    def save(self, out_dir, tokenizer=None):
        out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
        self.encoder.save_pretrained(out_dir)
        if tokenizer is not None: tokenizer.save_pretrained(out_dir)
        torch.save({t: h.state_dict() for t, h in self.heads.items()}, out_dir / 'heads.pt')
        with open(out_dir / 'label_maps.json', 'w', encoding='utf-8') as fh:
            json.dump({'base_model_name': self.base_model_name, 'head_labels': self.head_labels}, fh, indent=2)
    @classmethod
    def load(cls, model_dir, device='cpu'):
        model_dir = Path(model_dir)
        meta = json.load(open(model_dir / 'label_maps.json', encoding='utf-8'))
        model = cls(str(model_dir), meta['head_labels'])
        model.base_model_name = meta['base_model_name']
        head_state = torch.load(model_dir / 'heads.pt', map_location=device)
        for t, sd in head_state.items(): model.heads[t].load_state_dict(sd)
        return model.to(device).eval()

model = MultiTaskClassifier(BASE_MODEL, LABELS).to(device)
print('params (M):', round(sum(p.numel() for p in model.parameters()) / 1e6, 1))

## 7. Training loop
Label smoothing on all heads; sentiment uses inverse-frequency class weights **capped at 5x**.

In [ ]:
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score
from collections import Counter
from tqdm.auto import tqdm
import copy

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_dl) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)

ce = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scnt = Counter(train_df['sentiment'])
raw_w = [len(train_df) / (len(SENTIMENTS) * max(scnt.get(s, 0), 1)) for s in SENTIMENTS]
sw = torch.tensor([min(w, MAX_CLASS_WEIGHT) for w in raw_w], dtype=torch.float, device=device)
ce_sent = nn.CrossEntropyLoss(weight=sw, label_smoothing=LABEL_SMOOTHING)
loss_fns = {'category': ce, 'intent': ce, 'sentiment': ce_sent}
print('sentiment class weights (capped):', {s: round(float(w), 2) for s, w in zip(SENTIMENTS, sw)})

def evaluate(dl):
    model.eval()
    preds = {t: [] for t in TASKS}; gold = {t: [] for t in TASKS}
    with torch.no_grad():
        for batch in dl:
            ids = batch['input_ids'].to(device); am = batch['attention_mask'].to(device)
            logits = model(ids, am)
            for t in TASKS:
                preds[t] += logits[t].argmax(-1).cpu().tolist()
                gold[t]  += batch[t].tolist()
    return {t: (accuracy_score(gold[t], preds[t]), f1_score(gold[t], preds[t], average='macro')) for t in TASKS}

best_f1, best_state = -1.0, None
for epoch in range(1, EPOCHS + 1):
    model.train(); running = 0.0
    for batch in tqdm(train_dl, desc='epoch ' + str(epoch)):
        ids = batch['input_ids'].to(device); am = batch['attention_mask'].to(device)
        logits = model(ids, am)
        loss = sum(TASK_WEIGHTS[t] * loss_fns[t](logits[t], batch[t].to(device)) for t in TASKS)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        running += loss.item()
    m = evaluate(val_dl)
    macro = sum(m[t][1] for t in TASKS) / len(TASKS)
    summary = ' | '.join(t + ' acc=' + format(m[t][0], '.3f') + ' f1=' + format(m[t][1], '.3f') for t in TASKS)
    print('epoch', epoch, 'loss', round(running / len(train_dl), 4), '| val', summary, '| mean-f1', round(macro, 4))
    if macro > best_f1:
        best_f1 = macro; best_state = copy.deepcopy(model.state_dict())

if best_state is not None:
    model.load_state_dict(best_state)
print('best mean val macro-F1:', round(best_f1, 4))

## 8. Test-set evaluation
Category/intent are near-perfect because Bitext is clean and well-separated. The real test is the out-of-distribution set in the repo (`python eval/run_ood_eval.py`).

In [ ]:
from sklearn.metrics import classification_report

model.eval()
all_preds = {t: [] for t in TASKS}; all_gold = {t: [] for t in TASKS}
with torch.no_grad():
    for batch in test_dl:
        ids = batch['input_ids'].to(device); am = batch['attention_mask'].to(device)
        logits = model(ids, am)
        for t in TASKS:
            all_preds[t] += logits[t].argmax(-1).cpu().tolist()
            all_gold[t]  += batch[t].tolist()
for t in TASKS:
    print('==== ' + t + ' ====')
    ids = list(range(len(LABELS[t])))
    print(classification_report(all_gold[t], all_preds[t], labels=ids, target_names=LABELS[t], zero_division=0))

## 9. Save and zip the artifact

In [ ]:
import shutil

OUT_DIR = '/kaggle/working/insightflow_ticket_classifier'
model.save(OUT_DIR, tokenizer)
zip_path = shutil.make_archive('/kaggle/working/insightflow_ticket_classifier', 'zip', OUT_DIR)
print('saved model dir :', OUT_DIR)
print('zip for download:', zip_path)
print('files:', sorted(p.name for p in Path(OUT_DIR).iterdir()))

## 10. Inference sanity check
The phrasings that failed earlier OOD runs — a quick read on the fixes, and at what confidence.

In [ ]:
reloaded = MultiTaskClassifier.load(OUT_DIR, device=device)
tok = AutoTokenizer.from_pretrained(OUT_DIR)

samples = [
    'how do I change the email address on my profile',
    'need to update the card saved on my subscription',
    'I want my money back for the damaged item I returned',
    'love the product but the checkout page crashed when i paid',
    'wanted to say the new app update is fantastic',
    'I was charged twice for my subscription and Im annoyed',
]
enc = tok(samples, truncation=True, padding=True, max_length=MAX_LEN, return_tensors='pt').to(device)
with torch.no_grad():
    logits = reloaded(enc['input_ids'], enc['attention_mask'])
for i, s in enumerate(samples):
    print('TICKET:', s)
    for t in TASKS:
        probs = torch.softmax(logits[t][i], dim=-1)
        conf, idx = torch.max(probs, dim=-1)
        print('   ', t, '->', id2label[t][int(idx)], '(' + format(float(conf), '.2f') + ')')
    print()

## Next steps
1. Download `insightflow_ticket_classifier.zip` from the Kaggle Output panel.
2. **Delete** the old `models/insightflow_ticket_classifier/` and unzip this one in its place.
3. Re-run `python eval/run_ood_eval.py` and compare to `reports/ood_eval_v1.md`.

This is intended as the final training iteration: keep the calibration win (label smoothing), revert the sentiment regression (domain-neutral positives, capped class weight). After this we lock the model and write the evaluation report.